<a href="https://colab.research.google.com/github/KlyffHanger/TinyML/blob/main/ConvertingModel_TF_2_TFL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip install tensorflow
#!pip install tensorflow==2.20.0

import tensorflow as tf
print(tf.__version__)

# if tf.__version__ != "2.14.0":
#     print(f"Current TensorFlow version: {tf.__version__}, switching to 2.14.0")

#     # Uninstall current TensorFlow version
#     !pip uninstall -y tensorflow

#     # Install TensorFlow 2.10
#     !pip install numpy==1.26 --force-reinstall
#     !pip install tensorflow==2.14.0


#     # After installation, restart runtime
#     print("TensorFlow 2.14.0 installed.")
#     print("Please click on the Runtime > Restart session and run all.")
# else:
#     print("TensorFlow 2.14.0 is already installed.")

2.19.0


In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
print(f"Reconfirming the version ${tf.__version__}")

l0 = Dense(units=1, input_shape=[1])
model = Sequential([l0])
model.compile(optimizer='sgd', loss='mean_squared_error')

xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)

model.fit(xs, ys, epochs=500)

print(model.predict(np.array([10.0])))
print("Here is what I learned: {}".format(l0.get_weights()))

In [4]:
export_dir = 'saved_model/1'
model.export(export_dir)

Saved artifact at 'saved_model/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134560143739024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134560143740176: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [6]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()

In [ ]:
import pathlib
tflite_model_file = pathlib.Path('model.tflite')
tflite_model_file.write_bytes(tflite_model)

In [ ]:
#Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(input_details)
print(output_details)

In [ ]:
to_predict = np.array([[10.0]], dtype=np.float32)
print(to_predict)
interpreter.set_tensor(input_details[0]['index'], to_predict)
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])
print(tflite_results)

In [ ]:
import os

saved_model_size = sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(export_dir) for filename in filenames)
print(f"SavedModel directory size: {saved_model_size / (1024):.2f} KB")

tflite_model_size = os.path.getsize('model.tflite')
print(f"TFLite model file size: {tflite_model_size / (1024):.2f} KB")